# Qualitative examples on MiniImageNet

In [1]:
import os
import matplotlib.pyplot as plt
from matplotlib import rc
import torch
import numpy as np
from omegaconf import OmegaConf

from agreed.plots import plot_image, scale_to_original, plot_saliency_map_and_partition
from agreed.config.utils import get_config
from agreed.dataset.images.factory import ImageDatasetFactory
from agreed.dataset.images.utils import select_correctly_predicted_samples
from agreed.models.factory import ModelFactory

from agreed.explainers.images.decomposition import get_baseline, TorchModel, get_patch_partition
from agreed.explainers.imagenet.grouping import ImagenetAGREED

In [2]:
# Setup
rc('font',**{'size': 20})
device = "cuda:0"
np.set_printoptions(edgeitems=30, linewidth=100000, formatter=dict(float=lambda x: "%.3g" % x))
torch.manual_seed(0)

In [4]:
# Load the pre-trained model
# model_name = "ResNet18"
# model_name = "MobileNetV2"
model_name = "ConvNext"
device = "cpu"
batch_size = 128

# Load data
config = get_config("MiniImageNet", model_name)
print(OmegaConf.to_yaml(config))
_, test_dataset = ImageDatasetFactory().load_dataset(config.dataset)
# Keep transforms for later use when we plot [0, 1] images
image_transform = test_dataset.transforms.transform

model = ModelFactory().init_model(config.model, config.dataset).eval()
model = model.to(device)


save_path: /data/glaberge/agreed/MiniImageNet
data_path: /home/glaberge/Desktop/Pytorch/AGREED/experiments/MiniImageNet/datasets
model_path: /home/glaberge/Desktop/Pytorch/AGREED/experiments/MiniImageNet/models
model:
  name: ConvNext
  checkpoints: /home/glaberge/Desktop/Pytorch/AGREED/experiments/MiniImageNet/models/ConvNext/MiniImageNet/net.pth
  last_layer_name: classifier.2
dataset:
  name: MiniImageNet
  imsize:
  - 224
  - 224
  num_classes: 1000
  in_channels: 3
  path: /home/glaberge/Desktop/Pytorch/AGREED/experiments/MiniImageNet/datasets/MiniImageNet



In [ ]:
from agreed.dataset.images.utils import MINI_IMAGENET_CLS_TO_IMAGENET_NAME
# Only select samples from a given class
label = 24
print(f"Class {MINI_IMAGENET_CLS_TO_IMAGENET_NAME[label]}")
subset_test_dataset = select_correctly_predicted_samples(model, label, test_dataset, device, batch_size)

In [ ]:
instance_per_label = {
    0: 2,
    5: 0,
    24: 1,
}
image, y = subset_test_dataset[instance_per_label[label]]
with torch.no_grad():
    logits = model(image.unsqueeze(0).to(device)).cpu()
plt.figure(figsize=(20, 5))
plt.bar(range(100), height=logits[0])
plt.xticks(range(100), MINI_IMAGENET_CLS_TO_IMAGENET_NAME, rotation=90, fontsize=5)
plt.xlim(-1, 100)
plt.show()
plt.figure()
plot_image(image, image_transforms=image_transform)
plt.show()

In [ ]:
baseline = get_baseline("mean", config.dataset.imsize, config.dataset.in_channels, image.unsqueeze(0), model, device, label)
with torch.no_grad():
    logits = model(baseline.to(device)).cpu()
plt.figure(figsize=(20, 5))
plt.bar(range(100), height=logits[0])
plt.xticks(range(100), MINI_IMAGENET_CLS_TO_IMAGENET_NAME, rotation=90, fontsize=5)
plt.xlim(-1, 100)
# plt.ylim(-1, 7)
plt.show()
plt.figure()
plot_image(baseline[0], image_transforms=image_transform)
plt.show()

In [ ]:
# patch_size = 32
patch_size = 14
max_patch_size = 224*224/4
x = image.unsqueeze(0)
partition = get_patch_partition(patch_size, im_size=config.dataset.imsize)
class_1 = label
class_0 = 26

In [ ]:

thresholds = [0.9, 0.75, 0.5, 0.25]
init_partition = get_patch_partition(patch_size, config.dataset.imsize)
n_groups = None
iteration = 0
wrapped_model = TorchModel(
    model, 128, device, class_1, class_0
)

fig, axes = plt.subplots(len(thresholds), 3, figsize=((9, 3*len(thresholds))))

i = 0
for partition, (arch, occ) in ImagenetAGREED(
                    wrapped_model,
                    x,
                    baseline,
                    thresholds=thresholds,
                    partition=init_partition,
                    max_patch_size=max_patch_size,
                    verbose=True,
                ):
    n_groups = partition.D

    attributions = {"Arch": arch, "Occ": occ}

    disagreement = torch.sqrt(torch.sum((attributions["Arch"] - attributions["Occ"])**2))
    axes[i][0].set_ylabel(f"L ={disagreement: .2f}")

    scaled_image = scale_to_original(image, image_transforms=image_transform)
    plot_image(scaled_image, current_axis=axes[i][0])
    axes[i][0].set_title("Image")

    for j, (method, attribution) in enumerate(attributions.items()):
        current_axis = axes[i][j + 1]
        plot_saliency_map_and_partition(scaled_image, attribution, partition, method, current_axis, edge_color=(0, 0, 0))
        current_axis.set_title(method)
    i += 1
plt.show()

In [ ]:
print(arch)
print(occ)